## Data Cleaning and Validation

**Kebijakan**
- Baris yang melanggar aturan (kunci null, tanggal/bulan/tahun tidak konsisten, nilai negatif, NaN/Inf,
  hujan tidak monotonik) dibuang dan dihitung per tahap.
- Nilai ekstrem tidak dibuang: nilai tinggi pada hujan/debit adalah sinyal. Hanya diberi flag per kecamatan.
- Tidak ada imputasi: dataset tidak punya null.
- `flood_label` tidak dimodifikasi. Penyelidikan definisinya dilakukan di notebook terpisah (investigasi label/EDA).

In [1]:
import json
import os
from datetime import date
from pathlib import Path

import duckdb
import polars as pl

DATA_PATH = "../data/raw/flood_dataset.parquet"
OUT_PATH = "../data/processed/flood_clean.parquet"
MANIFEST_PATH = "../data/processed/cleaning_manifest.json"
assert os.path.exists(DATA_PATH), f"File tidak ditemukan: {DATA_PATH}"
Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

KEYS = ["district_id", "date"]
START, END = date(2015, 1, 30), date(2024, 12, 31)
EXPECTED_DAYS = (END - START).days + 1

RAIN = ["rain_1d", "rain_3d", "rain_7d", "rain_30d"]
DISC_LEVEL = ["disc_now", "disc_7d_mean"]
DISC_CHANGE = "disc_change_3d"
EXT_COLS = RAIN + DISC_LEVEL + [DISC_CHANGE]
LEVEL_COLS = RAIN + DISC_LEVEL

EPS = 1e-6      # toleransi floating point untuk perbandingan hujan
TOL = 0.011     # toleransi pembulatan 2 desimal untuk turunan debit
IQR_K = 1.5     # pengali IQR untuk flag disc_change_3d
UPPER_Q = 0.99  # persentil per kecamatan untuk flag kolom level 

raw = pl.scan_parquet(DATA_PATH)
schema = raw.collect_schema()
names = schema.names()
missing_cols = [c for c in KEYS + EXT_COLS + ["year", "month", "coastal", "flood_label"] if c not in names]
assert not missing_cols, f"Kolom tidak ada: {missing_cols}"
FLOATS = [c for c, t in schema.items() if t.is_float()]

manifest = {
    "sumber": DATA_PATH,
    "keluaran": OUT_PATH,
    "parameter": {"EPS": EPS, "TOL": TOL, "IQR_K": IQR_K, "UPPER_Q": UPPER_Q},
}

### 1. Kondisi awal

In [2]:
n_raw = raw.select(pl.len()).collect().item()
manifest["baris_mentah"] = n_raw
print(f"baris mentah = {n_raw:,}")
raw.null_count().collect().transpose(include_header=True, column_names=["jumlah_null"])

baris mentah = 1,152,432


column,jumlah_null
str,u32
"""district_id""",0
"""date""",0
"""year""",0
"""rain_1d""",0
"""rain_3d""",0
"""rain_7d""",0
"""rain_30d""",0
"""disc_now""",0
"""disc_7d_mean""",0


In [3]:
summary_awal = duckdb.sql(f"SUMMARIZE SELECT * FROM read_parquet('{DATA_PATH}')").pl()
summary_awal.select("column_name", "column_type", "min", "max", "avg", "count", "null_percentage")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,avg,count,null_percentage
str,str,str,str,str,i64,"decimal[9,2]"
"""district_id""","""VARCHAR""","""IDN.10.13.10_1""","""IDN.9.9.9_1""",null,1152432,0.00
"""date""","""TIMESTAMP""","""2015-01-30 00:00:00""","""2024-12-31 00:00:00""","""2020-01-15 12:00:00""",1152432,0.00
"""year""","""INTEGER""","""2015""","""2024""","""2019.5364238410596""",1152432,0.00
"""rain_1d""","""DOUBLE""","""0.0""","""258.7""","""6.793897687671181""",1152432,0.00
"""rain_3d""","""DOUBLE""","""0.0""","""415.70000000000005""","""20.382071566912654""",1152432,0.00
"""rain_7d""","""DOUBLE""","""0.0""","""645.2""","""47.561081174425716""",1152432,0.00
"""rain_30d""","""DOUBLE""","""0.0""","""1502.5""","""203.64353714578917""",1152432,0.00
"""disc_now""","""DOUBLE""","""0.0""","""1467.93""","""20.809406672146086""",1152432,0.00
"""disc_7d_mean""","""DOUBLE""","""0.0""","""1307.3185714285714""","""20.806564402186886""",1152432,0.00


### 2. Validasi aturan
Aturan untuk menentukan baris yang dibuang. Perbandingan hujan memakai toleransi `EPS` karena nilai
agregasi bergulir mengandung noise floating point.

In [4]:
base = raw.with_columns(pl.col("date").cast(pl.Date))


def not_binary(c):
    return ~pl.col(c).cast(pl.Float64).is_in([0.0, 1.0])


hard = {
    "key_null": pl.any_horizontal([pl.col(k).is_null() for k in KEYS]),
    "tanggal_di_luar_rentang": (pl.col("date") < START) | (pl.col("date") > END),
    "bulan_di_luar_1_12": ~pl.col("month").is_between(1, 12),
    "bulan_tidak_cocok_tanggal": pl.col("month") != pl.col("date").dt.month(),
    "tahun_tidak_cocok_tanggal": pl.col("year") != pl.col("date").dt.year(),
    "coastal_bukan_biner": not_binary("coastal"),
    "label_bukan_biner": not_binary("flood_label"),
    "hujan_1d_lebih_besar_3d": pl.col("rain_1d") > pl.col("rain_3d") + EPS,
    "hujan_3d_lebih_besar_7d": pl.col("rain_3d") > pl.col("rain_7d") + EPS,
    "hujan_7d_lebih_besar_30d": pl.col("rain_7d") > pl.col("rain_30d") + EPS,
}
for c in RAIN + DISC_LEVEL:
    hard[f"negatif_{c}"] = pl.col(c) < 0
for c in FLOATS:
    hard[f"nan_atau_inf_{c}"] = pl.col(c).is_nan() | pl.col(c).is_infinite()

viol = base.select([e.sum().alias(n) for n, e in hard.items()]).collect()
manifest["pelanggaran_aturan"] = viol.row(0, named=True)
viol.transpose(include_header=True, column_names=["pelanggaran"])

column,pelanggaran
str,u32
"""key_null""",0
"""tanggal_di_luar_rentang""",0
"""bulan_di_luar_1_12""",0
"""bulan_tidak_cocok_tanggal""",0
"""tahun_tidak_cocok_tanggal""",0
"""coastal_bukan_biner""",0
"""label_bukan_biner""",0
"""hujan_1d_lebih_besar_3d""",0
"""hujan_3d_lebih_besar_7d""",0


### 3. Duplikat

In [5]:
n_unique_keys = base.unique(subset=KEYS).select(pl.len()).collect().item()
per_district = base.group_by("district_id").agg(
    pl.len().alias("n_hari"),
    pl.col("coastal").n_unique().alias("n_nilai_coastal"),
).collect()
n_jam_bukan_nol = (
    raw.select((pl.col("date") != pl.col("date").dt.truncate("1d")).sum()).collect().item()
    if isinstance(schema["date"], pl.Datetime) else 0
)

panel = {
    "duplikat_kunci": n_raw - n_unique_keys,
    "kecamatan": per_district.height,
    "kecamatan_hari_tidak_lengkap": per_district.filter(pl.col("n_hari") != EXPECTED_DAYS).height,
    "kecamatan_coastal_tidak_konstan": per_district.filter(pl.col("n_nilai_coastal") > 1).height,
    "baris_jam_bukan_00": n_jam_bukan_nol,
    "baris_panel_diharapkan": per_district.height * EXPECTED_DAYS,
}
manifest["panel"] = panel
pl.DataFrame({"cek": list(panel), "nilai": list(panel.values())})

### 4. Validasi label terhadap debit
Catatan awal menyebut `flood_label` berasal dari debit di atas persentil ke-95 per kecamatan. Satu uji langsung:
apakah label sama dengan `disc_now > P95` pada hari yang sama. Penyelidikan lanjutan atas definisi label
(jendela ke depan, level persentil, sisa selisih) ada di notebook terpisah.

In [6]:
label_check = duckdb.sql(f"""
WITH thr AS (
    SELECT district_id, quantile_cont(disc_now, 0.95) AS p95
    FROM read_parquet('{DATA_PATH}')
    GROUP BY district_id
), j AS (
    SELECT r.disc_now, r.flood_label::INT AS y, t.p95
    FROM read_parquet('{DATA_PATH}') r
    JOIN thr t USING (district_id)
)
SELECT
    avg(y)                                          AS laju_label,
    avg((disc_now >  p95)::INT)                     AS laju_gt_p95,
    avg((disc_now >= p95)::INT)                     AS laju_ge_p95,
    avg(((disc_now >  p95)::INT = y)::INT)          AS cocok_gt_p95,
    avg(((disc_now >= p95)::INT = y)::INT)          AS cocok_ge_p95
FROM j
""").pl()
manifest["label_vs_p95"] = label_check.row(0, named=True)

lc = manifest["label_vs_p95"]
cocok_max = max(lc["cocok_gt_p95"], lc["cocok_ge_p95"])
baseline_nol = 1 - lc["laju_label"]
if cocok_max > 0.999:
    print(f"Label cocok {cocok_max:.4%} dengan ambang P95 pada hari yang sama.")
else:
    print(f"Label hanya cocok {cocok_max:.4%} dengan ambang P95 hari yang sama "
          f"(prediktor 'selalu 0' sudah {baseline_nol:.4%}): definisi label berbeda; "
          f"laju label {lc['laju_label']:.4%} vs laju P95 {lc['laju_gt_p95']:.4%}.")
label_check

Label hanya cocok 95.5766% dengan ambang P95 hari yang sama (prediktor 'selalu 0' sudah 92.6841%): definisi label berbeda; laju label 7.3159% vs laju P95 4.9709%.


laju_label,laju_gt_p95,laju_ge_p95,cocok_gt_p95,cocok_ge_p95
f64,f64,f64,f64,f64
0.073159,0.049709,0.053648,0.955766,0.952468


**Temuan.** Label tidak sama dengan `disc_now > P95` hari yang sama (cocok 95,58%; laju 7,32% vs 4,97%).
Label tidak diubah. Penyelidikan definisinya (hipotesis: debit melewati P95 kecamatan dalam 1-3 hari ke depan)
dicatat di notebook investigasi label, dan harus dipertimbangkan sebelum pemodelan.

In [7]:
# Nol debit per kecamatan (kualitas data: kecamatan tanpa sinyal debit)
zero_check = duckdb.sql(f"""
WITH per AS (
    SELECT district_id,
           avg(flood_label)              AS laju,
           avg((disc_now = 0)::INT)      AS nol,
           quantile_cont(disc_now, 0.95) AS p95,
           max(disc_now)                 AS maks
    FROM read_parquet('{DATA_PATH}')
    GROUP BY district_id
)
SELECT
    count(*)                                  AS n_kecamatan,
    count(*) FILTER (WHERE maks = 0)          AS kec_debit_selalu_nol,
    count(*) FILTER (WHERE p95 = 0)           AS kec_p95_nol,
    count(*) FILTER (WHERE nol > 0.05)        AS kec_nol_lebih_5persen,
    round(median(nol), 4)                     AS median_porsi_nol,
    round(max(nol), 4)                        AS maks_porsi_nol,
    round(min(laju), 4)                       AS laju_label_min,
    round(median(laju), 4)                    AS laju_label_median,
    round(max(laju), 4)                       AS laju_label_maks
FROM per
""").pl()
manifest["nol_debit"] = zero_check.row(0, named=True)

zero_ids = duckdb.sql(f"""
SELECT district_id FROM read_parquet('{DATA_PATH}')
GROUP BY district_id HAVING max(disc_now) = 0
""").pl()["district_id"].to_list()
manifest["kecamatan_debit_selalu_nol"] = zero_ids
print("kecamatan debit selalu 0:", zero_ids)
zero_check

kecamatan debit selalu 0: ['IDN.9.12.23_1']


n_kecamatan,kec_debit_selalu_nol,kec_p95_nol,kec_nol_lebih_5persen,median_porsi_nol,maks_porsi_nol,laju_label_min,laju_label_median,laju_label_maks
i64,i64,i64,i64,f64,f64,f64,f64,f64
318,1,1,22,0.0,1.0,0.0,0.0716,0.1032


### 5. Verifikasi kolom turunan debit
Memastikan `disc_change_3d` = `disc_now(t) - disc_now(t-3)` dan `disc_7d_mean` = rata-rata bergulir 7 hari.
Baris awal tiap kecamatan dilewati karena jendela dihitung dari data sebelum 2015-01-30.

In [8]:
idx = pl.int_range(pl.len()).over("district_id")
derived = (
    base.sort(KEYS)
    .with_columns(
        idx.alias("i"),
        (pl.col("disc_now") - pl.col("disc_now").shift(3).over("district_id")).alias("chg_hit"),
        pl.col("disc_now").rolling_mean(7).over("district_id").alias("m7_hit"),
    )
    .select(
        ((pl.col("i") >= 3) & ((pl.col(DISC_CHANGE) - pl.col("chg_hit")).abs() > TOL)).sum().alias("beda_change_3d"),
        ((pl.col("i") >= 6) & ((pl.col("disc_7d_mean") - pl.col("m7_hit")).abs() > TOL)).sum().alias("beda_mean_7d"),
        (pl.col("i") >= 6).sum().alias("baris_diuji"),
    )
    .collect()
)
manifest["verifikasi_turunan_debit"] = derived.row(0, named=True)
dv = manifest["verifikasi_turunan_debit"]
if dv["beda_change_3d"] == 0 and dv["beda_mean_7d"] == 0:
    print("Kedua kolom turunan konsisten penuh dengan disc_now: bukan informasi independen, dan tanpa kebocoran masa depan.")
else:
    print("Ada selisih pada kolom turunan: definisi berbeda dari asumsi, telusuri sebelum dipakai.")
derived

Kedua kolom turunan konsisten penuh dengan disc_now: bukan informasi independen, dan tanpa kebocoran masa depan.


beda_change_3d,beda_mean_7d,baris_diuji
u32,u32,u32
0,0,1150524


### 6. Flag ekstrem per kecamatan
- Kolom level (`rain_*`, `disc_now`, `disc_7d_mean`): flag = nilai di atas persentil ke-99 kecamatan tersebut
  (`UPPER_Q = 0.99`). Ini penanda ekor atas, bukan deteksi anomali: tingkat flag ≈1% menurut konstruksi.
- `disc_change_3d` (dapat naik dan turun): IQR dua sisi pada skala signed-log, `IQR_K = 1.5` (heuristik).
- Kuantil dihitung per kecamatan karena skala debit antar kecamatan berbeda jauh.
- IQR pada skala log tidak dipakai untuk kolom hujan/debit level: pada percobaan awal flagnya hanya muncul di ekor
  bawah (hujan mendekati nol di musim kering) dan ekor atas 0,0%.
- Tiap kolom punya flag sendiri (`ext_*`); `n_ekstrem` adalah jumlah kolom yang ekstrem pada baris itu.

In [9]:
def slog(c):
    return pl.col(c).sign() * pl.col(c).abs().log1p()


def ext_flag(c):
    if c in LEVEL_COLS:
        return (pl.col(c) > pl.col(c).quantile(UPPER_Q).over("district_id")).cast(pl.Int8)
    s = slog(c)
    q1 = s.quantile(0.25).over("district_id")
    q3 = s.quantile(0.75).over("district_id")
    iqr = q3 - q1
    return ((iqr > 0) & ((s > q3 + IQR_K * iqr) | (s < q1 - IQR_K * iqr))).cast(pl.Int8)


ext_rate = base.select(
    [(ext_flag(c).mean() * 100).round(2).alias(c) for c in EXT_COLS]
).collect()
manifest["persen_ekstrem_per_kolom"] = ext_rate.row(0, named=True)
ext_rate.transpose(include_header=True, column_names=["persen_baris_ekstrem"])

column,persen_baris_ekstrem
str,f64
"""rain_1d""",0.99
"""rain_3d""",0.99
"""rain_7d""",0.99
"""rain_30d""",0.99
"""disc_now""",0.98
"""disc_7d_mean""",0.99
"""disc_change_3d""",6.19


### 7. Pipeline cleaning
Urutan: buang pelanggaran aturan, deduplikasi kunci, casting tipe efisien, flag ekstrem, penanda kecamatan
debit nol, urutkan, tulis Parquet. Aturan yang bernilai null dianggap pelanggaran (`fill_null(True)`).

In [10]:
violates = pl.any_horizontal(list(hard.values())).fill_null(True)

step1 = base.filter(~violates)
step2 = step1.unique(subset=KEYS, keep="first", maintain_order=True)

clean = (
    step2.with_columns(
        pl.col("year").cast(pl.Int16),
        pl.col("month").cast(pl.Int8),
        pl.col("coastal").cast(pl.Int8),
        pl.col("flood_label").cast(pl.Int8),
    )
    .sort(KEYS)
    .with_columns([ext_flag(c).alias(f"ext_{c}") for c in EXT_COLS])
    .with_columns(
        pl.sum_horizontal([pl.col(f"ext_{c}") for c in EXT_COLS]).cast(pl.Int8).alias("n_ekstrem"),
        (pl.col("disc_now").max().over("district_id") == 0).cast(pl.Int8).alias("debit_selalu_nol"),
    )
)

n1 = step1.select(pl.len()).collect().item()
n2 = step2.select(pl.len()).collect().item()
clean_df = clean.collect()
clean_df.write_parquet(OUT_PATH, compression="zstd")

funnel = pl.DataFrame({
    "tahap": ["mentah", "setelah aturan", "setelah dedup", "final (ditulis)"],
    "baris": [n_raw, n1, n2, clean_df.height],
    "dibuang": [0, n_raw - n1, n1 - n2, n2 - clean_df.height],
})
manifest["funnel"] = funnel.to_dicts()
manifest["ukuran_keluaran_mb"] = round(os.path.getsize(OUT_PATH) / 1024**2, 2)
print(f"ditulis ke {OUT_PATH} | {manifest['ukuran_keluaran_mb']} MB")
funnel

ditulis ke ../data/processed/flood_clean.parquet | 14.6 MB


tahap,baris,dibuang
str,i64,i64
"""mentah""",1152432,0
"""setelah aturan""",1152432,0
"""setelah dedup""",1152432,0
"""final (ditulis)""",1152432,0


### 8. Verifikasi pasca-cleaning

In [11]:
chk = pl.scan_parquet(OUT_PATH)
post = chk.select([e.sum().alias(n) for n, e in hard.items()]).collect()
total_post = sum(post.row(0))
assert total_post == 0, f"Masih ada pelanggaran setelah cleaning: {total_post}"
assert chk.select(pl.len()).collect().item() == clean_df.height

final_stats = chk.select(
    pl.len().alias("baris"),
    pl.col("district_id").n_unique().alias("kecamatan"),
    pl.col("date").min().alias("awal"),
    pl.col("date").max().alias("akhir"),
    pl.col("flood_label").mean().alias("laju_label"),
    pl.col("n_ekstrem").mean().alias("rerata_n_ekstrem"),
    pl.col("debit_selalu_nol").sum().alias("baris_kecamatan_debit_nol"),
).collect()
manifest["pasca_cleaning"] = {k: str(v) for k, v in final_stats.row(0, named=True).items()}
print("pelanggaran aturan setelah cleaning:", total_post)
final_stats

pelanggaran aturan setelah cleaning: 0


baris,kecamatan,awal,akhir,laju_label,rerata_n_ekstrem,baris_kecamatan_debit_nol
u32,u32,date,date,f64,f64,i64
1152432,318,2015-01-30,2024-12-31,0.073159,0.121279,3624


### 9. DuckDB SUMMARIZE setelah cleaning

In [12]:
summary_akhir = duckdb.sql(f"SUMMARIZE SELECT * FROM read_parquet('{OUT_PATH}')").pl()
summary_akhir.select("column_name", "column_type", "min", "max", "avg", "std", "q25", "q50", "q75", "count", "null_percentage")

column_name,column_type,min,max,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,str,str,str,str,str,i64,"decimal[9,2]"
"""district_id""","""VARCHAR""","""IDN.10.13.10_1""","""IDN.9.9.9_1""",null,null,null,null,null,1152432,0.00
"""date""","""DATE""","""2015-01-30""","""2024-12-31""","""2020-01-15 12:00:00""",null,"""2017-07-23""","""2020-01-15""","""2022-07-10""",1152432,0.00
"""year""","""SMALLINT""","""2015""","""2024""","""2019.5364238410596""","""2.855671794074815""","""2017""","""2020""","""2022""",1152432,0.00
"""rain_1d""","""DOUBLE""","""0.0""","""258.7""","""6.793897687672233""","""10.101810641996664""","""0.3183446136854199""","""2.98025966296742""","""9.772820054585447""",1152432,0.00
"""rain_3d""","""DOUBLE""","""0.0""","""415.70000000000005""","""20.382071566913172""","""23.51372439659688""","""2.20377563129532""","""13.66482877747227""","""30.831075558986818""",1152432,0.00
"""rain_7d""","""DOUBLE""","""0.0""","""645.2""","""47.56108117442047""","""46.7152562955239""","""8.436215462500659""","""37.60499433579039""","""72.21623246743636""",1152432,0.00
"""rain_30d""","""DOUBLE""","""0.0""","""1502.5""","""203.6435371457939""","""165.60602301607733""","""60.15351905914371""","""179.97177026117612""","""310.3417064965559""",1152432,0.00
"""disc_now""","""DOUBLE""","""0.0""","""1467.93""","""20.8094066721514""","""71.059710983432""","""0.7273404155014092""","""2.7159045475033317""","""10.654373350874017""",1152432,0.00
"""disc_7d_mean""","""DOUBLE""","""0.0""","""1307.3185714285714""","""20.80656440218857""","""70.15783131833675""","""0.7683367466562574""","""2.786736083732891""","""10.83156044596608""",1152432,0.00


### 10. Manifest dan ringkasan temuan

In [13]:
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)

dibuang_total = n_raw - clean_df.height
lc = manifest["label_vs_p95"]
print(f"Baris: {n_raw:,} -> {clean_df.height:,} (dibuang {dibuang_total:,}).")
print(f"Total pelanggaran aturan pada data mentah: {sum(manifest['pelanggaran_aturan'].values()):,}.")
print(f"Duplikat kunci: {panel['duplikat_kunci']:,}; panel lengkap: "
      f"{panel['kecamatan_hari_tidak_lengkap'] == 0 and n_raw == panel['baris_panel_diharapkan']}.")
print(f"Kecocokan label vs P95: >={lc['cocok_ge_p95']:.4%}, >{lc['cocok_gt_p95']:.4%}; "
      f"laju label {lc['laju_label']:.4%}, laju P95 {lc['laju_gt_p95']:.4%}.")
print(f"Kecamatan dengan P95 debit = 0: {manifest['nol_debit']['kec_p95_nol']}.")
print(f"Manifest: {MANIFEST_PATH}")

Baris: 1,152,432 -> 1,152,432 (dibuang 0).
Total pelanggaran aturan pada data mentah: 0.
Duplikat kunci: 0; panel lengkap: True.
Kecocokan label vs P95: >=95.2468%, >95.5766%; laju label 7.3159%, laju P95 4.9709%.
Kecamatan dengan P95 debit = 0: 1.
Manifest: ../data/processed/cleaning_manifest.json


### Catatan Milestone 2
- `flood_label` bukan catatan kejadian banjir dan tidak sama dengan `disc_now > P95` pada hari yang sama
  (cocok 95,58%). Definisinya diselidiki terpisah; sampai itu ditetapkan, istilah saat ini adalah "label debit ekstrem",
  bukan "banjir". Rob dan penurunan muka tanah, yang relevan untuk kecamatan pesisir (89,9% data), tidak tertangkap.
- `disc_change_3d` dan `disc_7d_mean` adalah fungsi deterministik dari `disc_now` (selisih 0 pada seluruh baris
  yang diuji), bukan informasi independen.
- Satuan debit tidak terdokumentasi di sumber.
- 1 kecamatan berdebit selalu 0 (kolom `debit_selalu_nol`): dipertahankan di data, perlu dikeluarkan dari analisis
  spasial dan model. 22 kecamatan lain memiliki nol di atas 5% hari, sehingga ambang persentilnya kurang bermakna.
- Hanya 32 kecamatan non-pesisir; perbandingan pesisir vs non-pesisir berbasis sampel kecil.
- Flag ekstrem: kolom level memakai P99 per kecamatan (≈1% menurut konstruksi, bukan temuan); `disc_change_3d`
  memakai IQR k=1,5 yang heuristik. Kuantil dihitung atas seluruh periode, sehingga untuk pemodelan harus dihitung
  ulang dari data latih saja.
- Pipeline tidak membuang atau mengimputasi baris karena data lolos semua aturan.